In [ ]:
 # Runs 3 experiments


In [4]:
import numpy as np
from arguments import get_args
from dataset_loader import load_data  
from  feedforward_neural_network import FeedforwardNeuralNetwork
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


args=get_args()
dataset=args.dataset

X_train, Y_train, X_test, Y_test = load_data(dataset)

activation = args.activation                  # Activation function for hidden layers
optimizer = args.optimizer                  # Optimizer: "sgd", "momentum", "nag", "rmsprop", "adam", "nadam"
first_layer = X_train.shape[1]  # Integer
output_layer = Y_train.shape[1]  # Integer
num_layers = args.num_layers  # Integer
hidden_size = args.hidden_size  # Integer
epochs=args.epochs
learning_rate=args.learning_rate
weight_decay=args.weight_decay
batch_size=args.batch_size
weight_init=args.weight_init
# Construct the layers list correctly
layer_sizes = [first_layer] + [hidden_size] * num_layers + [output_layer]
layer_sizes = [784,128,128,10]


In [5]:
layer_sizes

[784, 128, 128, 10]

In [7]:

# Initialize the network using your FeedforwardNeuralNetwork class.
nn = FeedforwardNeuralNetwork(layer_sizes, activation, weight_init, learning_rate, weight_decay, batch_size, optimizer)
nn.train(X_train,Y_train,10,batch_size) 



Epoch 10/10, Loss: 0.2304, Accuracy: 0.1007


(0.2304237810195726, 0.10066666666666667)

In [ ]:

y_pred = nn.predict(X_test)
y_true = np.argmax(Y_test, axis=1)

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Define class names (for example, MNIST digits 0-9)
if args.dataset == "fashion_mnist":
    class_names = [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ]
else:  # Assume MNIST
    class_names = [str(i) for i in range(10)]
# Plot the confusion matrix using Seaborn
plt.figure(figsize=(8, 6))
ax = sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                 xticklabels=class_names, yticklabels=class_names)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix on Test Data")
plt.tight_layout()
plt.show()



In [10]:
import wandb
import numpy as np
from tensorflow.keras.datasets import fashion_mnist, mnist
from arguments import get_args

def cat_logging():
    # Get command-line arguments
    args = get_args()
    dataset = args.dataset

    # Set class names based on dataset
    if dataset == "fashion_mnist":
        class_names = [
            "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
            "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
        ]
    else:  # Assume MNIST
        class_names = [str(i) for i in range(10)]
    
    # Load dataset based on provided argument
    if dataset == "fashion_mnist":
        (X, y), (_, _) = fashion_mnist.load_data()
    elif dataset == "mnist":
        (X, y), (_, _) = mnist.load_data()
    else:
        raise ValueError("Dataset must be either 'mnist' or 'fashion_mnist'.")

    images_to_log = []

    # Loop through each unique category label and log the first occurrence
    unique_labels = np.unique(y)
    for label in unique_labels:
        idx = np.where(y == label)[0][0]
        image = X[idx]
        # Use the class_names list to get the proper label
        caption = class_names[label]
        images_to_log.append(wandb.Image(image, caption=caption))
    
    # Log all images under one key to W&B
    wandb.log({"Category Log": images_to_log})

import wandb
wandb.init(project="Fashion-MNIST-Sweep", entity="da24s019-indian-institute-of-technology-madras")
cat_logging()
wandb.finish()

In [ ]:
from feedforward_neural_network import FeedforwardNeuralNetwork
from dataset_loader import load_data
from arguments import get_args
import numpy as np
import wandb
from sweep_config import sweep_config  # Import the sweep config from a separate file
from train_sweep import train_sweep  # Import the training function

args=get_args()
if args.perform_sweep:
    # Initialize the W&B sweep (Only run this once per sweep)
    sweep_id = wandb.sweep(sweep_config, project="Fashion-MNIST-Sweep")
    print(f"Initialized sweep with ID: {sweep_id}")

    # Run multiple experiments using W&B Agent
    wandb.agent(sweep_id, function=train_sweep, count=50) 


dataset=args.dataset

X_train, Y_train,X_test, Y_test = load_data(dataset)

# Best parameters from sweep
layer_sizes = [784,128,128,128,10]         # Network architecture: input layer, one hidden layer, output layer
activation = "relu"                  # Activation function for hidden layers
weight_init = "random"               # Weight initialization method: "random", "xavier", etc.
learning_rate = 0.001                # Learning rate
weight_decay = 0.5                     # Weight decay 
batch_size = 32                      # Batch size
optimizer = "nadam"                   # Optimizer: "sgd", "momentum", "nag", "rmsprop", "adam", "nadam"
epochs = 10                          # Number of training epochs
args=get_args()
dataset = args.dataset
# Initialize the network using your FeedforwardNeuralNetwork class.
nn = FeedforwardNeuralNetwork(layer_sizes, activation, weight_init, learning_rate, weight_decay, batch_size, optimizer)
nn.train(X_train,Y_train,epochs,batch_size) 

y_pred_train = nn.predict(X_train)
y_true_train = np.argmax(Y_train, axis=1)


y_pred = nn.predict(X_test)
y_true = np.argmax(Y_test, axis=1)
    
args = get_args()

# Set class names based on the dataset
if args.dataset == "fashion_mnist":
    class_names = [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ]
else:  # Assume MNIST
    class_names = [str(i) for i in range(10)]




wandb.init(project="Fashion-MNIST-Sweep", entity="da24s019-indian-institute-of-technology-madras")


wandb.log({"conf_mat_test_data" : wandb.plot.confusion_matrix(probs=None,
                        y_true=y_true_train, preds=y_pred_train,
                        class_names=class_names)})

wandb.finish()

2025-03-09 18:11:58.347893: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741524118.368276   90199 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741524118.374637   90199 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-09 18:11:58.396045: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 10/10, Loss: 0.0491, Accuracy: 0.8323


Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: da24s019 (da24s019-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
